# Lecture 06 — Async crawling and politeness

> *"Doing 50 things at once is easy. Doing 50 things at once without burning down the server is the actual skill."*

The crawler from Lecture 04 fetches one page at a time. ~600ms per request × 1000 books = ~10 minutes. That's fine for `books.toscrape.com`. It's not fine for 100,000 pages.

The fix isn't *faster fetches* — they're already as fast as the network allows. The fix is to do many at once. That means **async**.

But concurrency without restraint turns your crawler into a small DoS attack. So this lecture is two things stitched together: how to run lots of requests at once, and how to *limit* yourself so you stay welcome.

## What you'll be able to do after this lecture

- Convert a sync crawler to an async one with minimal code change.
- Use `asyncio.Semaphore` to cap global concurrency.
- Implement per-host rate limiting so you can crawl multiple domains in parallel without hammering any one of them.
- Handle 429 responses correctly (the rare case where the *server* is right, you're wrong).

## Setup

```bash
pip install httpx beautifulsoup4 lxml aiosqlite
```


## 1. Why async, in one paragraph

A crawl is *I/O bound*: 99% of wall-clock time is waiting for bytes to arrive over the network. Threads can do the same job, but each thread has memory and context-switch overhead — Python tops out at maybe a few hundred. **`asyncio`** is cooperative: a single thread juggling thousands of in-flight requests by switching whenever one of them is waiting. For crawling that's the right shape.

## 2. The smallest async crawler

In [ ]:
import asyncio, httpx
from bs4 import BeautifulSoup

URLS = [f"https://books.toscrape.com/catalogue/page-{i}.html" for i in range(1, 6)]

async def fetch(client: httpx.AsyncClient, url: str) -> tuple[str, int]:
    r = await client.get(url, timeout=10.0)
    soup = BeautifulSoup(r.text, "lxml")
    return url, len(soup.select("article.product_pod"))


async def main():
    async with httpx.AsyncClient() as client:
        results = await asyncio.gather(*(fetch(client, u) for u in URLS))
    for url, n in results:
        print(f"{n:3d} products on {url.rsplit('/', 1)[-1]}")


# In a notebook the loop is already running:
await main()  # type: ignore[top-level-await]


Three things changed from the sync version:

- `httpx.Client` → `httpx.AsyncClient`. Methods are awaitable.
- `def fetch` → `async def fetch`. Calls have `await` in front of them.
- `for url in urls: fetch(url)` → `await asyncio.gather(*(fetch(u) for u in urls))`.

That's it. The same code now runs all five fetches concurrently.

## 3. Concurrency is not free — use a semaphore

`asyncio.gather(*lots_of_fetches)` will start them all *immediately*. If you have 10,000 URLs, that's 10,000 simultaneous TCP connections to one server. The server hates you. So do you, when your kernel runs out of file descriptors.

**Cap concurrency with `asyncio.Semaphore`.**

In [ ]:
SEM = asyncio.Semaphore(10)

async def fetch_capped(client: httpx.AsyncClient, url: str):
    async with SEM:
        r = await client.get(url)
        return url, r.status_code


Now no matter how many fetches you `gather`, only 10 are in flight at any moment. The rest are queued waiting their turn on the semaphore.

**Pick the number empirically.** Start small (5–10) and turn it up until either you hit diminishing returns or the target server starts unhappy-pinging you back.

## 4. Per-host rate limiting

A semaphore caps total concurrency. But if you're crawling three sites in one job, each site only sees a third of the load. What you actually want, often, is a *per-host* limit: "no more than 5 simultaneous requests to any one host, and a minimum 200ms gap between requests to the same host."

The cleanest pattern is a small dict-of-locks:

In [ ]:
import time, asyncio
from urllib.parse import urlparse


class HostThrottle:
    """Per-host: cap concurrency, enforce a minimum gap between requests."""

    def __init__(self, max_concurrent: int = 5, min_interval_s: float = 0.2):
        self.max_concurrent = max_concurrent
        self.min_interval_s = min_interval_s
        self._sems: dict[str, asyncio.Semaphore] = {}
        self._last: dict[str, float] = {}
        self._gate: dict[str, asyncio.Lock] = {}

    def _host(self, url: str) -> str:
        return urlparse(url).netloc

    def _sem(self, host: str) -> asyncio.Semaphore:
        if host not in self._sems:
            self._sems[host] = asyncio.Semaphore(self.max_concurrent)
            self._gate[host] = asyncio.Lock()
            self._last[host] = 0.0
        return self._sems[host]

    async def __aenter__url(self, url: str):  # not the cleanest API, see use below
        host = self._host(url)
        await self._sem(host).acquire()
        async with self._gate[host]:
            now = time.monotonic()
            wait = self._last[host] + self.min_interval_s - now
            if wait > 0:
                await asyncio.sleep(wait)
            self._last[host] = time.monotonic()

    def release(self, url: str):
        self._sems[self._host(url)].release()


# usage:
async def fetch_throttled(client, url, throttle):
    await throttle.__aenter__url(url)
    try:
        return await client.get(url)
    finally:
        throttle.release(url)


For real projects, libraries like [`aiometer`](https://pypi.org/project/aiometer/) or [`asyncio-throttle`](https://pypi.org/project/asyncio-throttle/) implement this pattern more cleanly. The shape, though, is the same: per-host semaphore + per-host last-request timestamp.

## 5. Backoff on 429

You're polite. You set a semaphore. You space out requests. Sometimes the server still says 429. **The server is right.** They have data you don't — about how loaded they are, how chatty other clients are, whether your IP is flagged. When you get a 429:

1. **Slow down.** Halve your effective rate.
2. **Honor `Retry-After`** if the response sets it (in seconds, or as an HTTP date).
3. **Don't queue thousands of retries** — that just resumes the storm.

A simple pattern:

In [ ]:
async def fetch_polite(client: httpx.AsyncClient, url: str, max_attempts: int = 5):
    for attempt in range(max_attempts):
        r = await client.get(url, timeout=15.0)
        if r.status_code == 429:
            retry_after = float(r.headers.get("retry-after", 2 ** attempt))
            await asyncio.sleep(retry_after)
            continue
        if r.status_code in (500, 502, 503, 504):
            await asyncio.sleep((2 ** attempt) + 0.5)
            continue
        return r
    return None  # give up


## 6. Async + SQLite

`sqlite3` is sync. Calling it from async code blocks the event loop, which defeats the whole point. Use **`aiosqlite`** for an async-native interface.

In [ ]:
import aiosqlite

async def init(db_path: str):
    async with aiosqlite.connect(db_path) as db:
        await db.execute("""CREATE TABLE IF NOT EXISTS pages (
            url TEXT PRIMARY KEY,
            html_size INTEGER,
            fetched_at TEXT
        )""")
        await db.commit()


async def store(db_path: str, url: str, size: int):
    async with aiosqlite.connect(db_path) as db:
        await db.execute(
            "INSERT OR REPLACE INTO pages VALUES (?, ?, datetime('now'))",
            (url, size),
        )
        await db.commit()


**One write-pool, not per-call.** Opening a fresh connection for every write is expensive. In real code, hold one `aiosqlite` connection for the lifetime of the crawl and pass it to your workers. Use a single writer task that pulls from a `Queue`, if you want to avoid contention entirely.

## 7. Putting it together — the async books crawler

Here's the Lecture 04 crawler, rewritten async, with a global semaphore and per-host throttle. Same shape, much faster.

In [ ]:
import asyncio, httpx, json, time, random
from urllib.parse import urljoin
from bs4 import BeautifulSoup

BASE = "https://books.toscrape.com/"
HEADERS = {
    "User-Agent": "CrawlingTutorial/0.1 (+https://github.com/Vladimir-125/CrawlingTutorial)",
}

GLOBAL_SEM = asyncio.Semaphore(20)


async def fetch(client, url, attempts=4):
    for i in range(attempts):
        async with GLOBAL_SEM:
            try:
                r = await client.get(url, timeout=15.0)
            except httpx.RequestError as e:
                last = e
            else:
                if r.status_code == 200:
                    return r
                if r.status_code in (404, 410):
                    return None
                last = httpx.HTTPStatusError("transient", request=r.request, response=r)
        await asyncio.sleep((2 ** i) + random.random())
    return None


async def parse_listing(client, url):
    r = await fetch(client, url)
    if r is None:
        return [], None
    soup = BeautifulSoup(r.text, "lxml")
    detail_urls = [
        urljoin(url, card.select_one("h3 a")["href"])
        for card in soup.select("article.product_pod")
    ]
    next_link = soup.select_one("li.next a")
    next_url = urljoin(url, next_link["href"]) if next_link else None
    return detail_urls, next_url


async def parse_detail(client, url):
    r = await fetch(client, url)
    if r is None:
        return None
    soup = BeautifulSoup(r.text, "lxml")
    title = soup.select_one("div.product_main h1").get_text(strip=True)
    price = soup.select_one("p.price_color").get_text(strip=True)
    return {"url": url, "title": title, "price": price}


async def crawl_books():
    started = time.monotonic()
    async with httpx.AsyncClient(headers=HEADERS, follow_redirects=True, http2=True) as client:
        # Walk listing pages sequentially (each tells us the next), parallelize the details
        url = urljoin(BASE, "catalogue/page-1.html")
        all_records = []
        while url:
            details, url = await parse_listing(client, url)
            results = await asyncio.gather(*(parse_detail(client, d) for d in details))
            all_records.extend(r for r in results if r)
        print(f"got {len(all_records)} books in {time.monotonic() - started:.1f}s")
        return all_records


# In a notebook:
# records = await crawl_books()


On a typical home connection, this finishes in around a minute, vs ~10 minutes for the sync version. The bottleneck is now mostly the listing-page chain, which is intrinsically sequential because each page tells you the next. To parallelize that too, you'd grab all 50 listing URLs up front (they're predictable here) — but that's a layout-specific optimization.

## 8. Patterns to internalize

- **Bound concurrency at the broadest reasonable level.** Per-host first; global as a backstop.
- **A jittered exponential backoff** is the right default for retries (`(2 ** attempt) + random()`). Without jitter, many simultaneous failures synchronize and re-storm the server.
- **One `AsyncClient` for the whole crawl.** Reuses connections.
- **HTTP/2** (`http2=True` on the client) multiplexes many requests over a single TCP connection. Helpful when allowed by the server.

## 9. Anti-patterns to avoid

- `for url in urls: await fetch(url)` — you've just rewritten sync code in async syntax. The `gather` (or task-pool) pattern is what makes it parallel.
- `asyncio.gather(*ten_thousand_tasks)` — you'll OOM the server, your kernel, or both. Use a semaphore.
- Sync I/O inside an async function (`requests.get`, `time.sleep`, blocking SQLite calls) — silently blocks the event loop. Use `httpx.AsyncClient`, `await asyncio.sleep`, `aiosqlite`.

## Recap

- Async is the right concurrency model for crawling — I/O-bound, lots of waiting.
- `httpx.AsyncClient` + `asyncio.gather` gets you parallelism in a few line changes.
- Cap with `asyncio.Semaphore`, both globally and per-host.
- Honor 429 with `Retry-After`, jittered exponential backoff for 5xx.
- Use async-native libraries (`aiosqlite`) when storing data — don't block the event loop.

## Exercises

1. Time the sync Lecture-04 crawler vs an async version with concurrency 5, 20, 50. Plot the speedup. Where does it level off?
2. Add a per-host throttle to your async crawler. Test that crawling two domains in parallel still keeps each under 5 concurrent.
3. Trigger a 429 on purpose by setting `concurrency=200` against `httpbin.org/anything`. Verify that your `Retry-After` handler kicks in.
4. Convert the JSONL+SQLite persistence from Lecture 04 to an async writer task. The fetcher tasks `put` onto a `Queue`; a single writer task `get`s and writes. Compare correctness against the version where every fetcher writes directly.

## Up next

**Lecture 07** — before you scrape another HTML page, *look for the API*. Sitemaps, JSON-LD, hidden XHR endpoints, RSS feeds. Half the pages you're tempted to scrape have a clean API one DevTools click away.
